# 11: ReportLab PDF Features Testing

This notebook tests comprehensive PDF generation features:

1. **Multi-page PDFs** - Multiple sections and page breaks
2. **Table of Contents** - Auto-generated TOC
3. **Charts in Reports** - Embedded matplotlib charts
4. **Styled Tables** - Tables with branding colors
5. **Headers/Footers** - Page numbers and branding

## Prerequisites

```bash
pip install reportlab pillow matplotlib
```

In [ ]:
import sys
from pathlib import Path
import pandas as pd
import numpy as np
from datetime import datetime
import tempfile
from IPython.display import display, HTML

# Ensure siege_utilities is importable
sys.path.insert(0, str(Path.cwd().parent))

import siege_utilities as su
su.configure_shared_logging(level="INFO")

print(f"Python: {sys.version}")
print("ReportLab PDF Features notebook ready.")

In [ ]:
# --- Output configuration ---
OUTPUT_DIR = Path("output") / "notebook_11"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# --- Branding configuration ---
from siege_utilities.reporting.client_branding import ClientBrandingManager
_branding_mgr = ClientBrandingManager()
BRANDING = _branding_mgr.get_client_branding('siege_analytics')
COLORS = BRANDING['colors']

print(f"Output directory: {OUTPUT_DIR}")
print(f"Branding: {BRANDING['name']} (primary={COLORS['primary']})")

In [ ]:
# Check ReportLab availability
try:
    from reportlab.platypus import SimpleDocTemplate, Paragraph
    from reportlab.lib.pagesizes import letter
    print("ReportLab: available")
except ImportError as e:
    print(f"ReportLab: NOT available ({e})")

try:
    from PIL import Image
    print("PIL (Pillow): available")
except ImportError as e:
    print(f"PIL: NOT available ({e})")

try:
    import matplotlib
    matplotlib.use('Agg')
    import matplotlib.pyplot as plt
    print(f"Matplotlib: available (backend: {matplotlib.get_backend()})")
except ImportError as e:
    print(f"Matplotlib: NOT available ({e})")

## 1. Basic PDF Generation

Test that basic PDF generation works.

In [ ]:
from siege_utilities.reporting.report_generator import ReportGenerator

# Initialize ReportGenerator — output goes to OUTPUT_DIR
rg = ReportGenerator(
    client_name="TestClient",
    output_dir=OUTPUT_DIR
)

print(f"ReportGenerator initialized (output: {OUTPUT_DIR})")

In [ ]:
# Create sample data
np.random.seed(42)

# Quarterly sales data
sales_data = pd.DataFrame({
    'Quarter': ['Q1 2025', 'Q2 2025', 'Q3 2025', 'Q4 2025'],
    'Revenue ($M)': [12.5, 15.2, 18.7, 22.1],
    'Growth (%)': [8.5, 12.1, 23.0, 18.2],
    'New Customers': [145, 189, 234, 278]
})

# Regional breakdown
regional_data = pd.DataFrame({
    'Region': ['Northeast', 'Southeast', 'Midwest', 'West', 'International'],
    'Revenue ($M)': [18.2, 12.5, 8.7, 22.4, 6.7],
    'Market Share (%)': [27, 18, 13, 33, 10]
})

print("Sales Data:")
display(sales_data)
print("\nRegional Data:")
display(regional_data)

In [ ]:
# Build basic report content
report_content = {
    'sections': [],
    'metadata': {
        'title': 'Quarterly Business Report',
        'subtitle': 'FY 2025 Performance Analysis',
        'author': 'Siege Analytics',
        'date': datetime.now().strftime('%B %d, %Y'),
        'client': 'TestClient'
    }
}

exec_summary = """
This report presents the quarterly performance analysis for FY 2025.
Key highlights include:

- Total revenue of $68.5M, up 15.4% year-over-year
- Strong growth in Q3 with 23% quarter-over-quarter increase
- 846 new customers acquired across all regions
- West region leads with 33% market share

The company is well-positioned for continued growth in FY 2026.
"""

report_content = rg.add_text_section(
    report_content,
    'Executive Summary',
    exec_summary
)

print("Report content built with executive summary.")

In [ ]:
# Add data tables
report_content = rg.add_table_section(report_content, 'Quarterly Performance', sales_data)
report_content = rg.add_table_section(report_content, 'Regional Breakdown', regional_data)

print(f"Report now has {len(report_content['sections'])} sections.")

In [ ]:
# Generate basic PDF
timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
basic_pdf_path = OUTPUT_DIR / f"basic_report_{timestamp}.pdf"

try:
    rg.generate_pdf_report(report_content, str(basic_pdf_path))
    
    if basic_pdf_path.exists():
        size_kb = basic_pdf_path.stat().st_size / 1024
        print(f"Basic PDF generated: {basic_pdf_path.name} ({size_kb:.1f} KB)")
    else:
        print("WARNING: PDF was not created")
except Exception as e:
    print(f"ERROR: {e}")
    import traceback
    traceback.print_exc()

## 2. Multi-Page PDF with Page Breaks

Test multi-page PDF generation with explicit page breaks.

In [ ]:
# Create multi-page report
multipage_content = {
    'sections': [],
    'metadata': {
        'title': 'Comprehensive Annual Report',
        'subtitle': 'Multi-Section Analysis',
        'author': 'Siege Analytics',
        'date': datetime.now().strftime('%B %d, %Y'),
    }
}

# Section 1: Introduction (with page break after)
intro_text = """
This comprehensive report covers multiple aspects of business performance.
The analysis spans financial metrics, market position, customer analysis,
and strategic recommendations.

Report Structure:
1. Financial Performance
2. Market Analysis
3. Customer Insights
4. Strategic Recommendations
"""
multipage_content = rg.add_section(
    multipage_content, 'text', 'Introduction', intro_text, page_break_after=True
)

# Section 2: Financial Performance
financial_text = """
The financial performance in FY 2025 exceeded expectations across all key metrics.
Revenue grew by 15.4% compared to FY 2024, driven primarily by expansion
in the West region and successful product launches.

Key Financial Highlights:
- Total Revenue: $68.5M
- Gross Margin: 72.3%
- Operating Income: $18.2M
- Free Cash Flow: $12.8M
"""
multipage_content = rg.add_text_section(multipage_content, 'Financial Performance', financial_text)
multipage_content = rg.add_table_section(multipage_content, 'Quarterly Financials', sales_data)
multipage_content['sections'][-1]['page_break_after'] = True

# Section 3: Market Analysis
market_text = """
Market analysis reveals strong competitive positioning across all regions.
The company has successfully expanded market share in key territories
while maintaining premium pricing power.
"""
multipage_content = rg.add_text_section(multipage_content, 'Market Analysis', market_text)
multipage_content = rg.add_table_section(multipage_content, 'Regional Market Share', regional_data)

# Section 4: Recommendations
recommendations_text = """
Strategic priorities for FY 2026:

1. Continue investment in West region growth initiatives
2. Expand international presence, targeting European markets
3. Launch customer retention program to reduce churn
4. Increase R&D spending to maintain technology leadership
5. Explore strategic acquisitions in adjacent markets
"""
multipage_content = rg.add_text_section(multipage_content, 'Strategic Recommendations', recommendations_text)

print(f"Multi-page report built with {len(multipage_content['sections'])} sections.")

In [ ]:
# Generate multi-page PDF
multipage_pdf_path = OUTPUT_DIR / f"multipage_report_{timestamp}.pdf"

try:
    rg.generate_pdf_report(multipage_content, str(multipage_pdf_path))
    
    if multipage_pdf_path.exists():
        size_kb = multipage_pdf_path.stat().st_size / 1024
        print(f"Multi-page PDF generated: {multipage_pdf_path.name} ({size_kb:.1f} KB)")
    else:
        print("WARNING: PDF was not created")
except Exception as e:
    print(f"ERROR: {e}")
    import traceback
    traceback.print_exc()

## 3. Table of Contents Template

Test the dedicated Table of Contents template.

In [ ]:
from reportlab.pdfgen import canvas
from reportlab.lib.pagesizes import letter
from siege_utilities.reporting.templates.table_of_contents_template import (
    TableOfContentsTemplate,
    create_table_of_contents
)

# Define TOC sections
toc_sections = [
    {
        'title': 'Executive Summary', 'page': 3,
        'subsections': [
            {'title': 'Key Findings', 'page': 3},
            {'title': 'Recommendations', 'page': 4}
        ]
    },
    {
        'title': 'Financial Performance', 'page': 5,
        'subsections': [
            {'title': 'Revenue Analysis', 'page': 5},
            {'title': 'Cost Structure', 'page': 6},
            {'title': 'Profitability Metrics', 'page': 7}
        ]
    },
    {
        'title': 'Market Analysis', 'page': 8,
        'subsections': [
            {'title': 'Competitive Landscape', 'page': 8},
            {'title': 'Market Share Trends', 'page': 9}
        ]
    },
    {'title': 'Strategic Recommendations', 'page': 10},
    {'title': 'Appendices', 'page': 11}
]

print(f"TOC defined: {len(toc_sections)} main sections")

In [ ]:
# Generate TOC PDF
toc_pdf_path = OUTPUT_DIR / f"table_of_contents_{timestamp}.pdf"

try:
    c = canvas.Canvas(str(toc_pdf_path), pagesize=letter)
    create_table_of_contents(
        canvas_obj=c, sections=toc_sections,
        title="Table of Contents", page_number=2
    )
    c.save()
    
    if toc_pdf_path.exists():
        size_kb = toc_pdf_path.stat().st_size / 1024
        print(f"TOC PDF generated: {toc_pdf_path.name} ({size_kb:.1f} KB)")
    else:
        print("WARNING: PDF was not created")
except Exception as e:
    print(f"ERROR: {e}")
    import traceback
    traceback.print_exc()

## 4. Charts in Reports

Test embedding matplotlib charts in PDF reports.

In [ ]:
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

# Create sample charts using branding colors
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Bar chart: Revenue by quarter
quarters = sales_data['Quarter']
revenue = sales_data['Revenue ($M)']
bar_colors = [COLORS['primary'], COLORS['secondary'], COLORS.get('accent', '#FF6B35'), COLORS.get('background', '#F5F5F5')]

axes[0].bar(quarters, revenue, color=bar_colors)
axes[0].set_title('Quarterly Revenue', fontsize=14, fontweight='bold')
axes[0].set_ylabel('Revenue ($M)')
axes[0].set_xlabel('Quarter')
for i, v in enumerate(revenue):
    axes[0].text(i, v + 0.5, f'${v}M', ha='center', fontsize=10)

# Pie chart: Regional market share
regions = regional_data['Region']
market_share = regional_data['Market Share (%)']
pie_colors = [COLORS['primary'], COLORS['secondary'], COLORS.get('accent', '#FF6B35'), COLORS.get('text_color', '#333333'), COLORS.get('header_footer_text_color', '#FFFFFF')]

axes[1].pie(market_share, labels=regions, autopct='%1.0f%%', colors=pie_colors, startangle=90)
axes[1].set_title('Regional Market Share', fontsize=14, fontweight='bold')

plt.tight_layout()

# Save chart for embedding in PDF
chart_path = OUTPUT_DIR / f"charts_{timestamp}.png"
plt.savefig(chart_path, dpi=150, bbox_inches='tight', facecolor='white')

# Show inline in notebook before closing
plt.show()
plt.close()

print(f"Chart saved: {chart_path.name} ({chart_path.stat().st_size / 1024:.1f} KB)")

In [ ]:
# Create report with embedded chart
from siege_utilities.reporting.templates.base_template import BaseReportTemplate
from reportlab.platypus import Paragraph, Spacer, PageBreak
from reportlab.platypus import Image as RLImage
from reportlab.lib.units import inch

chart_report_path = OUTPUT_DIR / f"report_with_charts_{timestamp}.pdf"

try:
    template = BaseReportTemplate(str(chart_report_path))
    story = []
    
    # Title page
    story.extend(template.add_title_page(
        title="Business Analytics Report",
        subtitle="With Embedded Charts",
        author="Siege Analytics",
        date=datetime.now().strftime('%B %d, %Y')
    ))
    
    # Introduction + chart
    story.append(Paragraph("Data Visualizations", template.styles['Heading1']))
    story.append(Spacer(1, 12))
    story.append(Paragraph(
        "The following charts illustrate key business metrics including quarterly "
        "revenue trends and regional market share distribution.",
        template.styles['Normal']
    ))
    story.append(Spacer(1, 24))
    
    if chart_path.exists():
        img = RLImage(str(chart_path), width=7*inch, height=3.5*inch)
        story.append(img)
        story.append(Spacer(1, 12))
        story.append(Paragraph(
            "Figure 1: Quarterly Revenue and Regional Market Share",
            template.styles.get('Caption', template.styles['Normal'])
        ))
    
    template.build_document(story)
    
    if chart_report_path.exists():
        size_kb = chart_report_path.stat().st_size / 1024
        print(f"Chart report PDF generated: {chart_report_path.name} ({size_kb:.1f} KB)")
    else:
        print("WARNING: PDF was not created")
except Exception as e:
    print(f"ERROR: {e}")
    import traceback
    traceback.print_exc()

## 5. Comprehensive Report with All Features

Test a full report combining all features.

In [ ]:
from reportlab.platypus import Table, TableStyle
from reportlab.lib import colors

comprehensive_report_path = OUTPUT_DIR / f"comprehensive_report_{timestamp}.pdf"

try:
    template = BaseReportTemplate(str(comprehensive_report_path))
    story = []
    
    # Title page
    story.extend(template.add_title_page(
        title="Annual Business Report",
        subtitle="FY 2025 Comprehensive Analysis",
        author="Siege Analytics",
        date=datetime.now().strftime('%B %d, %Y')
    ))
    
    # Executive Summary
    story.append(Paragraph("Executive Summary", template.styles['Heading1']))
    story.append(Spacer(1, 12))
    story.append(Paragraph(
        "This comprehensive report presents the financial and operational performance "
        "for FY 2025. The company achieved strong results across all key metrics, "
        "with revenue growing 15.4% year-over-year to $68.5M.",
        template.styles['Normal']
    ))
    story.append(Spacer(1, 24))
    
    # Quarterly Performance Table
    story.append(Paragraph("Quarterly Performance", template.styles['Heading2']))
    story.append(Spacer(1, 12))
    
    table_data = [sales_data.columns.tolist()] + sales_data.values.tolist()
    t = Table(table_data)
    t.setStyle(TableStyle([
        ('BACKGROUND', (0, 0), (-1, 0), colors.HexColor(COLORS['primary'])),
        ('TEXTCOLOR', (0, 0), (-1, 0), colors.white),
        ('ALIGN', (0, 0), (-1, -1), 'CENTER'),
        ('FONTNAME', (0, 0), (-1, 0), 'Helvetica-Bold'),
        ('FONTSIZE', (0, 0), (-1, 0), 11),
        ('BOTTOMPADDING', (0, 0), (-1, 0), 12),
        ('TOPPADDING', (0, 0), (-1, 0), 12),
        ('BACKGROUND', (0, 1), (-1, -1), colors.HexColor(COLORS.get('background', '#F5F5F5'))),
        ('TEXTCOLOR', (0, 1), (-1, -1), colors.black),
        ('FONTNAME', (0, 1), (-1, -1), 'Helvetica'),
        ('FONTSIZE', (0, 1), (-1, -1), 10),
        ('GRID', (0, 0), (-1, -1), 0.5, colors.HexColor('#CCCCCC')),
        ('VALIGN', (0, 0), (-1, -1), 'MIDDLE'),
        ('BOTTOMPADDING', (0, 1), (-1, -1), 8),
        ('TOPPADDING', (0, 1), (-1, -1), 8),
    ]))
    story.append(t)
    story.append(Spacer(1, 24))
    
    # Embedded chart
    story.append(Paragraph("Data Visualizations", template.styles['Heading2']))
    story.append(Spacer(1, 12))
    if chart_path.exists():
        img = RLImage(str(chart_path), width=6*inch, height=3*inch)
        story.append(img)
    story.append(Spacer(1, 12))
    story.append(PageBreak())
    
    # Regional Analysis
    story.append(Paragraph("Regional Analysis", template.styles['Heading1']))
    story.append(Spacer(1, 12))
    story.append(Paragraph(
        "The regional breakdown shows strong performance across all territories. "
        "The West region continues to lead with 33% market share.",
        template.styles['Normal']
    ))
    story.append(Spacer(1, 24))
    
    regional_table_data = [regional_data.columns.tolist()] + regional_data.values.tolist()
    t2 = Table(regional_table_data)
    t2.setStyle(TableStyle([
        ('BACKGROUND', (0, 0), (-1, 0), colors.HexColor(COLORS['secondary'])),
        ('TEXTCOLOR', (0, 0), (-1, 0), colors.white),
        ('ALIGN', (0, 0), (-1, -1), 'CENTER'),
        ('FONTNAME', (0, 0), (-1, 0), 'Helvetica-Bold'),
        ('FONTSIZE', (0, 0), (-1, 0), 11),
        ('BOTTOMPADDING', (0, 0), (-1, 0), 12),
        ('TOPPADDING', (0, 0), (-1, 0), 12),
        ('BACKGROUND', (0, 1), (-1, -1), colors.HexColor(COLORS.get('background', '#F5F5F5'))),
        ('TEXTCOLOR', (0, 1), (-1, -1), colors.black),
        ('FONTNAME', (0, 1), (-1, -1), 'Helvetica'),
        ('FONTSIZE', (0, 1), (-1, -1), 10),
        ('GRID', (0, 0), (-1, -1), 0.5, colors.HexColor('#CCCCCC')),
        ('VALIGN', (0, 0), (-1, -1), 'MIDDLE'),
        ('BOTTOMPADDING', (0, 1), (-1, -1), 8),
        ('TOPPADDING', (0, 1), (-1, -1), 8),
    ]))
    story.append(t2)
    story.append(Spacer(1, 24))
    
    # Recommendations
    story.append(Paragraph("Strategic Recommendations", template.styles['Heading2']))
    story.append(Spacer(1, 12))
    for i, rec in enumerate([
        "Continue investment in West region growth initiatives",
        "Expand international presence targeting European markets",
        "Launch customer retention program to reduce churn",
        "Increase R&D spending to maintain technology leadership",
        "Explore strategic acquisitions in adjacent markets"
    ], 1):
        story.append(Paragraph(f"{i}. {rec}", template.styles['Normal']))
        story.append(Spacer(1, 6))
    
    template.build_document(story)
    
    if comprehensive_report_path.exists():
        size_kb = comprehensive_report_path.stat().st_size / 1024
        print(f"Comprehensive PDF generated: {comprehensive_report_path.name} ({size_kb:.1f} KB)")
    else:
        print("WARNING: PDF was not created")
except Exception as e:
    print(f"ERROR: {e}")
    import traceback
    traceback.print_exc()

## Summary

List all generated PDFs.

In [ ]:
# Summary — check all generated files
print("=" * 50)
print("REPORTLAB PDF FEATURE TEST RESULTS")
print("=" * 50)

generated_files = [
    ('Basic PDF', basic_pdf_path),
    ('Multi-page PDF', multipage_pdf_path),
    ('Table of Contents', toc_pdf_path),
    ('Charts Report', chart_report_path),
    ('Comprehensive Report', comprehensive_report_path),
]

results = []
for name, path in generated_files:
    if path.exists():
        size_kb = path.stat().st_size / 1024
        print(f"  PASS  {name}: {size_kb:.1f} KB")
        results.append(True)
    else:
        print(f"  FAIL  {name}: Not generated")
        results.append(False)

passed = sum(results)
total = len(results)
print(f"\n{passed}/{total} PDFs generated successfully.")
print(f"Output directory: {OUTPUT_DIR}")

if passed == total:
    print("\nAll ReportLab PDF features working!")
else:
    print("\nSome features need attention.")